<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/lab-03-an-mcp-server-for-cobalt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 (graded) — An MCP server for Cobalt
**Course 3: AI Agents and Agentic AI with Python — Chapter 3: The Model Context Protocol (MCP)**

**Problem brief (Sam Okafor, Cobalt Manufacturing):** "Every team wraps the same plant
systems in their own bespoke tool code. IT wants one standard way to expose equipment data
and runbooks to any assistant."

**What you'll submit:** a custom "equipment registry" MCP server (≥ 2 tools, ≥ 1 resource)
connected to an agent, answering maintenance questions through it, with the trust boundary
documented. Uses the real `mcp` Python SDK over stdio — already fully local/offline.

In [ ]:
!pip install -q mcp

## 1. The equipment dataset

In [ ]:
import json

equipment_registry = {
    'PMP-014': {'name': 'Feed Pump 14', 'status': 'running', 'last_service': '2026-06-01',
                'runbook': 'Check inlet pressure first; if below 2 bar, inspect the suction strainer for clogging.'},
    'CNV-002': {'name': 'Conveyor Belt 2', 'status': 'stopped', 'last_service': '2026-07-15',
                'runbook': 'A stopped state after an E-stop requires a manual reset at the local panel before restart.'},
    'PRS-007': {'name': 'Hydraulic Press 7', 'status': 'running', 'last_service': '2026-05-20',
                'runbook': 'Unusual noise during the compression stroke typically indicates low hydraulic fluid.'},
}
with open('equipment_registry.json', 'w') as f:
    json.dump(equipment_registry, f)

## 2. Write the MCP server to a file
This is the real `mcp` Python SDK's `FastMCP` — the exact pattern used in production MCP
servers. `equipment_status` and `lookup_runbook` are **tools**; `equipment://<id>` is a
**resource**.

In [ ]:
server_code = '''
import json
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("cobalt-equipment-registry")

with open("equipment_registry.json") as f:
    REGISTRY = json.load(f)

@mcp.tool()
def equipment_status(equipment_id: str) -> str:
    """Look up the current status and last-service date for a piece of equipment by ID."""
    eq = REGISTRY.get(equipment_id)
    if not eq:
        return f"No equipment found with ID {equipment_id}"
    return f"{eq[\'name\']}: status={eq[\'status\']}, last serviced {eq[\'last_service\']}"

@mcp.tool()
def lookup_runbook(equipment_id: str) -> str:
    """Return the maintenance runbook guidance for a piece of equipment by ID."""
    eq = REGISTRY.get(equipment_id)
    if not eq:
        return f"No runbook found for {equipment_id}"
    return eq["runbook"]

@mcp.resource("equipment://{equipment_id}")
def equipment_resource(equipment_id: str) -> str:
    """Expose the full equipment record as a readable resource."""
    return json.dumps(REGISTRY.get(equipment_id, {}))

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open('equipment_server.py', 'w') as f:
    f.write(server_code)
print('Server written to equipment_server.py')

## 3. Connect a client over stdio and call the tools

In [ ]:
import sys

async def run_mcp_demo():
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    server_params = StdioServerParameters(command=sys.executable, args=['equipment_server.py'])
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print('Tools exposed by the server:', [t.name for t in tools.tools])

            status_result = await session.call_tool('equipment_status', {'equipment_id': 'PMP-014'})
            print('equipment_status(PMP-014):', status_result.content[0].text)

            runbook_result = await session.call_tool('lookup_runbook', {'equipment_id': 'CNV-002'})
            print('lookup_runbook(CNV-002):', runbook_result.content[0].text)

            resource = await session.read_resource('equipment://PRS-007')
            print('resource equipment://PRS-007:', resource.contents[0].text)
            return True

try:
    mcp_worked = await run_mcp_demo()
except Exception as e:
    print(f'Real MCP client/server demo unavailable in this runtime ({e}).')
    print('Offline fallback — calling the same tool functions directly (same contract, no protocol layer):')
    import importlib.util
    spec = importlib.util.spec_from_file_location('equipment_server_direct', 'equipment_server.py')
    # can't import a script with mcp.run() at module level directly without the SDK; simulate instead
    with open('equipment_registry.json') as f:
        REGISTRY = json.load(f)
    print('equipment_status(PMP-014):', f"{REGISTRY['PMP-014']['name']}: status={REGISTRY['PMP-014']['status']}")
    print('lookup_runbook(CNV-002):', REGISTRY['CNV-002']['runbook'])
    mcp_worked = False

## 4. Connect an agent to the server and answer a maintenance question

In [ ]:
def answer_maintenance_question(equipment_id):
    """A minimal agent loop (Chapter 1's pattern): one status check, one runbook lookup,
    then compose an answer. Uses direct calls here for notebook simplicity — in Chapter 1's
    full loop, these would be the two tool calls an LLM chooses."""
    with open('equipment_registry.json') as f:
        registry = json.load(f)
    eq = registry.get(equipment_id)
    if not eq:
        return f'No record for {equipment_id}.'
    return (f"{eq['name']} is currently {eq['status']} (last serviced {eq['last_service']}). "
            f"Guidance: {eq['runbook']}")

print(answer_maintenance_question('PMP-014'))

## 5. Trust boundary (fill in)
This server is code you wrote and trust. If Cobalt's IT team connected to a **third-party**
MCP server instead, what would you want to verify before letting an agent use it — and what's
the specific risk if a tool's `description` string (not just its output) turned out to be
written maliciously? (Full treatment: Chapter 9.)

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 3: The Model Context Protocol (MCP)*